## Filtrado de 100 documentos cualquiera

In [4]:
from pathlib import Path
import re

ruta = Path(r"C:\notebooks_IR\datos\gutenberg")

# Textos originales y documentos tokenizados.
textos = {}
corpus = {}

def normalizar_texto(texto):
    return re.findall(r"[^\W\d_]+", texto.lower(), flags=re.UNICODE)

for archivo in sorted(ruta.rglob("*.txt")):
    # Identificar el libro para evitar distintas versiones del mismo.
    coincidencia = re.fullmatch(r"pg(\d+)(?:-\d+)?", archivo.stem)
    if not coincidencia:
        continue

    doc_id = coincidencia.group(1)

    if doc_id in corpus:
        continue

    texto = archivo.read_text(encoding="utf-8", errors="replace")
    tokens = normalizar_texto(texto)

    if not tokens:
        continue

    textos[doc_id] = texto
    corpus[doc_id] = tokens

    if len(corpus) == 100:
        break

if len(corpus) != 100:
    raise ValueError(f"Solo se encontraron {len(corpus)} documentos válidos.")

print("Documentos:", len(corpus))
print("Total de tokens:", sum(len(tokens) for tokens in corpus.values()))

for doc_id in list(corpus)[:5]:
    print(f"Libro {doc_id}: {len(corpus[doc_id]):,} tokens")

Documentos: 100
Total de tokens: 7345721
Libro 1: 4,806 tokens
Libro 10: 795,234 tokens
Libro 100: 991,578 tokens
Libro 1000: 105,167 tokens
Libro 10000: 17,446 tokens


## Matríz Término Frecuencia

In [5]:
from collections import Counter
import pandas as pd

# Contar las apariciones de cada término por documento.
frecuencias = {
    doc_id: Counter(tokens)
    for doc_id, tokens in corpus.items()
}

# Crear la matriz y completar con 0 los términos ausentes.
matriz_tf = (
    pd.DataFrame.from_dict(frecuencias, orient="columns")
    .fillna(0)
    .astype("int64")
    .sort_index()
)

matriz_tf.index.name = "termino"
matriz_tf.columns.name = "documento"

print("Dimensiones (términos, documentos):", matriz_tf.shape)
display(matriz_tf.iloc[:20, :10])

Dimensiones (términos, documentos): (113452, 100)


documento,1,10,100,1000,10000,10001,10002,10003,10004,10005
termino,,,,,,,,,,
a,82,8235,16548,2107,207,189,1427,1507,1368,1555
aa,0,0,0,0,0,0,0,0,0,0
aad,0,0,0,0,0,0,0,0,0,0
aaraaf,0,0,0,0,0,0,0,0,0,0
aaron,0,350,99,0,0,0,0,0,0,0
aaronites,0,2,0,0,0,0,0,0,0,0
ab,0,0,0,1,0,0,0,0,0,0
aba,0,0,0,0,0,0,0,0,0,0
aback,0,0,0,0,0,1,0,1,0,0


## Vector IDF

In [6]:
import numpy as np

# Número de documentos.
N = matriz_tf.shape[1]

# Frecuencia documental: contar documentos donde el término aparece.
df = (matriz_tf > 0).sum(axis=1)

# Vector IDF, conservando los términos como índice.
vector_idf = np.log(N / df)
vector_idf.name = "idf"

print("Vector IDF:")
display(vector_idf.to_frame().head(20))


Vector IDF:


,idf
termino,
a,0.000000
aa,3.506558
aad,4.605170
aaraaf,4.605170
aaron,2.659260
aaronites,4.605170
ab,2.120264
aba,4.605170
aback,2.207275


## Matriz TF-IDF

In [7]:
# Multiplicar cada fila de TF por el IDF de su término.
matriz_tfidf = matriz_tf.mul(vector_idf, axis="index")

print("Matriz TF-IDF:", matriz_tfidf.shape)
display(matriz_tfidf.iloc[:20, :10])

Matriz TF-IDF: (113452, 100)


documento,1,10,100,1000,10000,10001,10002,10003,10004,10005
termino,,,,,,,,,,
a,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
aa,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
aad,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
aaraaf,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
aaron,0.0,930.741013,263.266744,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
aaronites,0.0,9.210340,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
ab,0.0,0.000000,0.000000,2.120264,0.0,0.000000,0.0,0.000000,0.0,0.0
aba,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
aback,0.0,0.000000,0.000000,0.000000,0.0,2.207275,0.0,2.207275,0.0,0.0


Ver terminos con mayor peso

In [8]:
doc_id = matriz_tfidf.columns[0]

print(f"Términos con mayor peso en el documento {doc_id}:")
display(
    matriz_tfidf[doc_id]
    .sort_values(ascending=False)
    .head(15)
    .to_frame(name="tf_idf")
)

Términos con mayor peso en el documento 1:


,tf_idf
termino,
declaration,15.249238
facsimiles,10.519674
usurpations,10.519674
independence,10.183725
etexts,9.210340
abolishing,8.987197
colonies,7.864451
etext,5.991465
tape,5.898339
